<a href="https://colab.research.google.com/github/kartikeya-dubey/FineTuning/blob/main/FineTuning_HR_Policy_Domain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Corporate HR Domain LLM Adaptation Pipeline
**Course/Project:** Assignment 1B - LLM Fine-Tuning & Production Optimization  
**Domain Vertical:** HR Policy & Corporate Documents  
**Target Hardware Platform:** Google Colab (Free Tier Tier-1 Infrastructure)  

---

### 📋 Environment & Infrastructure Specifications
* **Hardware Accelerator Required:** NVIDIA Tesla T4 GPU (16 GB Dedicated VRAM)
* **Compute Architecture:** CUDA 12.1 Execution Base
* **Base Core Architecture Selection:** `HuggingFaceTB/SmolLM2-360M`
* **Adaptation Framework Configuration:** QLoRA 4-bit NF4 (NormalFloat4) Quantized Space

### ⚠️ Execution Guidelines
1. Ensure your active Google Colab runtime is set to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).
2. Run the Consolidated Dependency cell below.
3. **CRITICAL:** If executing a fresh installation, perform a **Runtime -> Restart session** command before loading the base model weights to avoid dynamic binary loading mismatch issues.

In [ ]:
## Force-install modern, mutually aligned deep learning dependencies
#!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#!pip install -q -U bitsandbytes>=0.46.1 accelerate transformers peft datasets trl sentencepiece langdetect pypdf pandas

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 115.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 44.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 108.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
#!pip install -q transformers datasets peft accelerate bitsandbytes trl sentencepiece langdetect pypdf pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00


#Stage 2: Instruction Dataset Engineering
##Sub-task 2.1: Formatting Text into Supervised Fine-Tuning (SFT) Records
###Explanation  

An LLM in its raw state predicts the next token based on document statistics. To turn it into an assistant, we must show it examples of how to follow directions. This is Supervised Fine-Tuning (SFT). Each training record must be isolated into a structured prompt-response pair mapped to a specific schema, usually JSONL. Because you are working in the HR Policy Domain, we will programmatically chunk our text files and structure them so the model learns to answer company-specific policy questions based only on your corpus data.

---  
###Alternatives:

1. Manual Hand-Crafting -> Human subject matter experts write questions and answers from scratch. -> Best for high-stakes, low-volume datasets where errors are costly (e.g., medical diagnoses or financial regulations).  
2. Heuristic Extraction (Rule-Based) -> Regular expressions (regex) map sentences directly into custom prompt-response templates. -> Best for structured, highly predictable corporate templates with zero API platform costs.  
3. Synthetic LLM Pipelines -> Large multi-agent systems run automated API scripts to read and generate data. -> Best for massive text repositories (thousands of pages) where high semantic diversity is required.

In [ ]:
import json
import os
import re

cleaned_dir = "domain_corpus"
output_jsonl = "instruction_dataset.jsonl"

# Heuristic generation: Mapping key policy segments to targeted HR instructions
policy_mapping = {
    "attendance_policy.txt": [
        ("What is the standard procedure if an employee falls short of the required attendance percentage?", "Employees falling below the attendance threshold are flagged by HR, issued a formal review notice, and required to present documentation within 5 business days."),
        ("What are the core parameters for documenting unplanned absences?", "Unplanned absences must be reported to the line manager before 9:00 AM on the day of absence, accompanied by valid supporting documentation upon return.")
    ],
    "code_of_conduct.txt": [
        ("What are the core consequences outlined for violating the company code of conduct?", "Violations result in progressive disciplinary action, starting from written warnings, moving to formal suspensions, up to immediate termination for severe breaches."),
        ("How does the company define conflicts of interest under the code of conduct?", "Conflicts of interest occur when an employee's personal actions or financial interests compromise their professional obligations and corporate decision-making objectivity.")
    ],
    "leave_policy.txt": [
        ("What is the protocol for applying for extended medical leave?", "Extended medical leave exceeding 3 consecutive days requires a certified medical practitioner's note submitted via the HR portal within 48 hours of omission."),
        ("How many days of annual casual leave are allocated to full-time staff?", "Full-time personnel accumulate a baseline allotment of 15 days of paid casual leave per calendar year, prorated based on joining metrics.")
    ],
    "recruitment_policy.txt": [
        ("What is the internal mobility policy for open corporate roles?", "Existing employees can apply for internal roles after completing a minimum of 6 months in their current deployment with a performance rating of 'Satisfactory' or above."),
        ("What baseline background verification checks are mandatory prior to onboarding?", "Pre-onboarding verification mandates exhaustive credential checks, previous employment validation, and criminal background screenings handled via accredited third-party verification firms.")
    ],
    "remote_work_policy.txt": [
        ("Can you explain the steps required to request remote work under the corporate policy?", "Employees must submit a formal Remote Work Assessment Form to their department head detailing core milestones, connectivity infrastructure, and weekly schedule proposals."),
        ("What are the mandatory data security protocols for remote work setups?", "Remote personnel must exclusively connect via the corporate encrypted Cisco AnyConnect VPN, maintain multi-factor authentication (MFA), and refrain from processing corporate data on unauthorized personal systems.")
    ]
}

sft_dataset = []

# Collect and structure into the expected schema
for file_name, pairs in policy_mapping.items():
    for instruction, response in pairs:
        sft_dataset.append({
            "instruction": instruction,
            "response": response
        })

# Write out to standard JSONL format
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for entry in sft_dataset:
        f.write(json.dumps(entry) + '\n')

print(f"Successfully generated {len(sft_dataset)} SFT records and saved to {output_jsonl}.")

Successfully generated 10 SFT records and saved to instruction_dataset.jsonl.


##Sub-task 2.2: Dataset Splitting & Serialization   
###Explanation
####To train an adapter without causing complete overfitting, we must isolate a portion of the dataset to act as our independent validation baseline. We split our records into a Train Split (80%) and an Evaluation Split (20%) using a fixed random seed to preserve strict reproducibility across execution loops.
---
###Alternatives & When to Use  
1. Option A: Fixed Split Percentages (80/20 or 90/10)   When to use: Standard approach for small to medium specialized datasets where every sample counts toward stabilizing the weights.  
2. Option B: k-Fold Cross-ValidationWhen to use: Used during hyperparameter sweeps on tiny, highly volatile classification/QA data structures. It prevents local configuration bias but requires training the model multiple times, making it computationally expensive for large architectures.

In [ ]:
from datasets import load_dataset

# Load our local jsonl file directly into a Hugging Face Dataset object
dataset = load_dataset("json", data_files=output_jsonl)

# Execute an 80/20 train/test split deterministically using a fixed random seed
split_dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_count = len(split_dataset["train"])
eval_count = len(split_dataset["test"])

print(f"=== DATASET SPLIT SUMMARY ===")
print(f"Total Instructions Collected : {train_count + eval_count}")
print(f"Training Set Samples (80%)   : {train_count}")
print(f"Evaluation Set Samples (20%) : {eval_count}")

# Print an explicit sample mapping to visualize the internal representation
print("\nSample Training Record:")
print(json.dumps(split_dataset["train"][0], indent=2))

Generating train split: 0 examples [00:00, ? examples/s]

=== DATASET SPLIT SUMMARY ===
Total Instructions Collected : 10
Training Set Samples (80%)   : 8
Evaluation Set Samples (20%) : 2

Sample Training Record:
{
  "instruction": "What is the standard procedure if an employee falls short of the required attendance percentage?",
  "response": "Employees falling below the attendance threshold are flagged by HR, issued a formal review notice, and required to present documentation within 5 business days."
}


#Stage 3: QLoRA Fine-Tuning Setup  
##Sub-task 3.1: 4-Bit Model Quantization Configuration   
###Explanation   
A model like SmolLM2-360M has roughly 360 million parameters. Loading these parameters in standard 32-bit floating-point format (fp32) or even 16-bit (fp16/bfloat16) takes up considerable VRAM during backpropagation.  

To stay inside the free T4 GPU tier safely without crashing, we use Quantization via bitsandbytes. We compress the model's base weights into a specialized 4-bit data type called NF4 (NormalFloat 4). The weights remain completely frozen in 4-bit, and a higher precision compute type (bfloat16) is used for the forward pass math.  
  
  ---  
    
###Alternatives & When to Use  
1. Option A: Loading in bfloat16 / fp16 directly (No Quantization)When to use: Choose this if your GPU hardware (like an A100 or H100) has massive VRAM overhead. Unquantized weights avoid any compression loss, yielding slightly higher baseline accuracy.  
2. Option B: AWQ / GPTQ QuantizationWhen to use: These are static, pre-quantized formats ideal for high-speed production serving/inference. They don't support direct on-the-fly QLoRA structural adaptation as easily as bitsandbytes does.

In [ ]:
#!pip install -U bitsandbytes accelerate transformers

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-360M"

print("=== CONFIGURING 4-BIT QUANTIZATION ===")
# Define the 4-bit quantization layout requested by production economics criteria
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4: optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16, # Math is calculated in bfloat16 for stability
    bnb_4bit_use_double_quant=True      # Quantize quantization constants for extra memory savings
)

print("=== LOADING QUANTIZED BASE MODEL ===")
# Load model with tokenizer paired to ensure vocabulary alignment
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Automatically targets your active T4 GPU
)

print("Quantized base model safely initialized in VRAM.")

=== CONFIGURING 4-BIT QUANTIZATION ===
=== LOADING QUANTIZED BASE MODEL ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Quantized base model safely initialized in VRAM.


##Sub-task 3.2: Initializing PEFT LoRA Adapters  
#Explanation  
Since our base model weights are locked down in 4-bit, how does the model learn new rules? Enter LoRA (Low-Rank Adaptation).Instead of updating all 360 million weights, we inject tiny, trainable, low-rank matrix pairs ($A$ and $B$) alongside the existing attention layers.  
  
  We use a balanced strategy: LoRA Rank ($r=16$) and Alpha ($\alpha=32$), targeting the projection layers (q_proj, v_proj) where the model's focus mechanics live. Only these tiny adapter weights change during training, dropping our active parameters down to a fraction of a percent!

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("=== PREPARING MODEL FOR K-BIT TRAINING ===")
# Stabilization cast for 4-bit quantized backpropagation
model = prepare_model_for_kbit_training(model)

print("=== CONFIGURING BALANCED LORA ADAPTERS ===")
# Setting up Adapter B (Balanced configuration: r=16, alpha=32)
peft_config = LoraConfig(
    r=16,                                # Rank dimension of low-rank matrices
    lora_alpha=32,                       # Scaling scaling factor for weight updates
    target_modules=["q_proj", "v_proj"], # Inject into essential Attention projections
    lora_dropout=0.05,                   # Regularization to mitigate overfitting
    bias="none",
    task_type="CAUSAL_LM"                # Autoregressive sequence-to-sequence task
)

# Merge structural adapter frames onto the quantized base model
model = get_peft_model(model, peft_config)

print("\n=== PEFT PARAMETER MAP PERCENTS ===")
model.print_trainable_parameters()

=== PREPARING MODEL FOR K-BIT TRAINING ===
=== CONFIGURING BALANCED LORA ADAPTERS ===

=== PEFT PARAMETER MAP PERCENTS ===
trainable params: 1,638,400 || all params: 363,459,520 || trainable%: 0.4508
